# ChordFlow — Sağ El Model Eğitimi

Bu notebook sağ el için `Unknown (0) + Class 1–5` classifier modelini eğitir.

- Input: 21 landmark × XYZ = 63 float
- Normalizasyon: wrist merkezleme, wrist–middle MCP ölçekleme, XY rotasyon hizalama, train mean/std
- Split: ardışık kamera karelerinin sızmasını azaltan temporal bloklar
- Model: `63 → 128 → 64 → 6 logits`
- Çıktı: PyTorch, TorchScript, ONNX, metrik CSV/JSON ve grafikler

Class 1–5 yalnızca dinamik kimliklerdir; akor isimleri React tarafında sonradan eşlenir.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "model" / "data" / "right" / "landmarks.csv").exists():
    if (PROJECT_ROOT / "data" / "right" / "landmarks.csv").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("model/data/right/landmarks.csv bulunamadı.")

MODEL_DIR = PROJECT_ROOT / "model"
CSV_PATH = MODEL_DIR / "data" / "right" / "landmarks.csv"
OUTPUT_ROOT = MODEL_DIR / "training_outputs" / "right_hand"

print(f"Proje: {PROJECT_ROOT}")
print(f"Dataset: {CSV_PATH}")

## 1. Dataset kontrolü

Sınıfların dengesi ve MediaPipe handedness confidence değerleri incelenir. `0`, explicit Unknown sınıfıdır.

In [ ]:
frame = pd.read_csv(CSV_PATH)
class_counts = frame["label"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot.bar(ax=axes[0], color="#7c3aed")
axes[0].set(title="Sağ el sınıf dağılımı", xlabel="Class", ylabel="Örnek sayısı")
axes[0].grid(axis="y", alpha=0.25)

frame.boxplot(column="handedness_score", by="label", ax=axes[1])
axes[1].set(title="MediaPipe handedness confidence", xlabel="Class", ylabel="Confidence")
fig.suptitle("")
fig.tight_layout()
plt.show()

display(class_counts.rename("sample_count").to_frame())

## 2. Eğitim

Aşağıdaki hücre `right_hand_model_training.py` scriptini çalıştırır. Script:

1. Ham 63 koordinatı geometrik olarak normalize eder.
2. Veriyi sınıf başına temporal bloklarla train/validation/test olarak böler.
3. Train split üzerinden standardizasyon değerlerini hesaplar.
4. MLP'yi augmentation, class-weight, dropout, scheduler ve early stopping ile eğitir.
5. En iyi validation checkpoint'ini test split üzerinde değerlendirir.

In [ ]:
command = [
    sys.executable,
    str(MODEL_DIR / "right_hand_model_training.py"),
    "--csv", str(CSV_PATH),
    "--output", str(OUTPUT_ROOT),
]
result = subprocess.run(command, cwd=MODEL_DIR, check=True, text=True)
print("Eğitim tamamlandı.")

## 3. Sonuçlar

En son eğitim run'ının metrikleri, sınıf raporu ve tanı grafikleri yüklenir.

In [ ]:
run_dirs = sorted(OUTPUT_ROOT.glob("run_*"))
if not run_dirs:
    raise FileNotFoundError("Eğitim çıktısı bulunamadı. Önce eğitim hücresini çalıştırın.")

LATEST_RUN = run_dirs[-1]
metrics = json.loads((LATEST_RUN / "metrics.json").read_text(encoding="utf-8"))
report = pd.read_csv(LATEST_RUN / "classification_report.csv", index_col=0)
thresholds = pd.read_csv(LATEST_RUN / "confidence_thresholds.csv")

print(f"Run: {LATEST_RUN.name}")
print(f"Test accuracy: {metrics['test_accuracy']:.2%}")
print(f"Test macro F1: {metrics['test_macro_f1']:.2%}")
print(f"Best epoch: {metrics['best_epoch']}")
display(report.round(4))
display(thresholds.round(4))

In [ ]:
for filename in [
    "training_history.png",
    "confusion_matrix.png",
    "confidence_analysis.png",
]:
    print(filename)
    display(Image(filename=str(LATEST_RUN / filename)))

## 4. ONNX export ve doğrulama

Preprocessing ONNX graph içine gömülür. Tarayıcı doğrudan ham `1 × 63` MediaPipe landmark dizisi gönderir. Doğrulama scripti PyTorch ve ONNX logits farkını ölçer.

In [ ]:
subprocess.run(
    [sys.executable, str(MODEL_DIR / "export_left_hand_onnx.py"), "--hand", "right"],
    cwd=MODEL_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, str(MODEL_DIR / "verify_left_hand_onnx.py"), "--hand", "right"],
    cwd=MODEL_DIR,
    check=True,
)
print("Sağ el ONNX modeli hazır: public/models/right_hand_model.onnx")